# 02 - Nettoyage et journal des décisions

In [1]:
import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
from src.chargement import charger_brut

commandes, articles, clients, produits, traduction = charger_brut()
journal = []

## 1. Dates de livraison manquantes

In [2]:
sans_livraison = commandes["order_delivered_customer_date"].isna()
print(sans_livraison.sum())
print(commandes.loc[sans_livraison, "order_status"].value_counts())
journal.append(("Dates de livraison vides conservées",
                "Commandes non livrées (annulées, en cours)", int(sans_livraison.sum())))

2965
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


## 2. Catégorie produit manquante

In [3]:
nb = int(produits["product_category_name"].isna().sum())
produits["product_category_name"] = produits["product_category_name"].fillna("inconnu")
journal.append(("Catégorie vide remplacée par inconnu", "L'absence est une information", nb))

## 3. Traduction des catégories

In [4]:
produits = produits.merge(traduction, on="product_category_name", how="left", validate="many_to_one")
nb_sans_trad = int(produits["product_category_name_english"].isna().sum())
produits["product_category_name_english"] = (produits["product_category_name_english"]
                                             .fillna(produits["product_category_name"]))
journal.append(("Catégories sans traduction : nom portugais conservé",
                "Catégories absentes du fichier de traduction", nb_sans_trad))

## 4. Doublons

In [5]:
nb_exacts = int(articles.duplicated().sum())
nb_cle = int(articles.duplicated(subset=["order_id", "order_item_id"]).sum())
print(nb_exacts, nb_cle)
articles = articles.drop_duplicates(subset=["order_id", "order_item_id"])
journal.append(("Doublons supprimés sur (order_id, order_item_id)", "Clé unique d'un article", nb_cle))

0 0


## 5. Valeurs aberrantes sur le prix

In [6]:
q1, q3 = articles["price"].quantile([0.25, 0.75])
borne_haute = q3 + 1.5 * (q3 - q1)
nb_chers = int((articles["price"] > borne_haute).sum())
print(borne_haute, nb_chers, articles["price"].max())
journal.append(("Prix élevés conservés", "Aberrant ne veut pas dire faux : produits chers réels", nb_chers))

277.4 8427 6735.0


## 6. Types

In [7]:
clients["customer_state"] = clients["customer_state"].astype("category")
commandes["order_status"] = commandes["order_status"].astype("category")
journal.append(("customer_state et order_status en category", "Variables à peu de valeurs", len(clients)))

## Journal de nettoyage

In [8]:
pd.DataFrame(journal, columns=["Quoi", "Pourquoi", "Lignes touchées"])

,Quoi,Pourquoi,Lignes touchées
0,Dates de livraison vides conservées,"Commandes non livrées (annulées, en cours)",2965
1,Catégorie vide remplacée par inconnu,L'absence est une information,610
2,Catégories sans traduction : nom portugais con...,Catégories absentes du fichier de traduction,623
3,"Doublons supprimés sur (order_id, order_item_id)",Clé unique d'un article,0
4,Prix élevés conservés,Aberrant ne veut pas dire faux : produits cher...,8427
5,customer_state et order_status en category,Variables à peu de valeurs,99441


In [9]:
sortie = Path("../data/processed")
sortie.mkdir(parents=True, exist_ok=True)
commandes.to_csv(sortie / "commandes_propre.csv", index=False)
articles.to_csv(sortie / "articles_propre.csv", index=False)
produits.to_csv(sortie / "produits_propre.csv", index=False)
clients.to_csv(sortie / "clients_propre.csv", index=False)
print("fichiers propres enregistrés")

fichiers propres enregistrés
